# Dynamic Hedging with DCC-GARCH
Hedge ratios and effectiveness for S&P 500 spot vs E-mini futures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import yfinance as yf
from arch import arch_model
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Chart style
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Crimson  = '#DC3545'
GoldC    = '#DAA520'

mpl.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

In [ ]:
# Download S&P 500 (spot) and E-mini futures
tickers = ['^GSPC', 'ES=F']

data = yf.download(tickers, start='2004-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close']
data.columns = ['Spot', 'Futures']
data = data.dropna()

# Percentage returns
returns = data.pct_change().dropna() * 100

print(f'Sample: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Observations: {len(returns)}')
print(f'Correlation: {returns.corr().iloc[0,1]:.4f}')

In [ ]:
# Fit GARCH(1,1) for both series
garch_res = {}
cond_vol = {}
std_resid = {}

for name in ['Spot', 'Futures']:
    am = arch_model(returns[name].dropna(), vol='Garch', p=1, q=1,
                    mean='Constant', dist='t')
    res = am.fit(disp='off')
    garch_res[name] = res
    cond_vol[name] = res.conditional_volatility
    std_resid[name] = res.std_resid
    print(f'{name}: alpha={res.params["alpha[1]"]:.4f}, '
          f'beta={res.params["beta[1]"]:.4f}')

# DCC estimation
z_df = pd.DataFrame({'Spot': std_resid['Spot'],
                      'Futures': std_resid['Futures']}).dropna()
z = z_df.values  # T x 2
T, k = z.shape

# Unconditional correlation
Q_bar = np.corrcoef(z.T)
rho_bar = Q_bar[0, 1]
print(f'\nUnconditional correlation (CCC): {rho_bar:.4f}')

# DCC log-likelihood
def dcc_loglik(params, z, Q_bar):
    a, b = params
    if a < 0 or b < 0 or a + b >= 1:
        return 1e10
    T, k = z.shape
    Q_t = Q_bar.copy()
    ll = 0.0
    for t in range(T):
        Q_t = (1 - a - b) * Q_bar + a * np.outer(z[t], z[t]) + b * Q_t
        d = np.sqrt(np.diag(Q_t))
        R_t = Q_t / np.outer(d, d)
        try:
            sign, logdet = np.linalg.slogdet(R_t)
            if sign <= 0:
                return 1e10
            R_inv = np.linalg.inv(R_t)
            ll += -0.5 * (logdet + z[t] @ R_inv @ z[t] - z[t] @ z[t])
        except np.linalg.LinAlgError:
            return 1e10
    return -ll

res_dcc = minimize(dcc_loglik, x0=[0.02, 0.95], args=(z, Q_bar),
                   method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-8})
a_dcc, b_dcc = res_dcc.x
print(f'DCC parameters: a = {a_dcc:.6f}, b = {b_dcc:.6f}')

In [ ]:
# Compute hedge ratios
# DCC hedge ratio: h_t = rho_t * sigma_spot_t / sigma_fut_t
# CCC hedge ratio: h_t = rho_bar * sigma_spot_t / sigma_fut_t
# OLS static: h = beta from regression spot = alpha + beta * futures

vol_df = pd.DataFrame({'Spot': cond_vol['Spot'],
                        'Futures': cond_vol['Futures']}).loc[z_df.index]
ret_aligned = returns.loc[z_df.index]

# DCC time-varying correlation
rho_dcc = np.zeros(T)
Q_t = Q_bar.copy()
for t in range(T):
    Q_t = (1 - a_dcc - b_dcc) * Q_bar + a_dcc * np.outer(z[t], z[t]) + b_dcc * Q_t
    d = np.sqrt(np.diag(Q_t))
    R_t = Q_t / np.outer(d, d)
    rho_dcc[t] = R_t[0, 1]

sigma_spot = vol_df['Spot'].values
sigma_fut  = vol_df['Futures'].values

# DCC hedge ratio
h_dcc = rho_dcc * sigma_spot / sigma_fut

# CCC hedge ratio
h_ccc = rho_bar * sigma_spot / sigma_fut

# OLS static hedge ratio
beta_ols = np.polyfit(ret_aligned['Futures'].values,
                      ret_aligned['Spot'].values, 1)[0]
h_ols = beta_ols

print(f'OLS static hedge ratio: {h_ols:.4f}')
print(f'CCC hedge ratio range: [{h_ccc.min():.4f}, {h_ccc.max():.4f}]')
print(f'DCC hedge ratio range: [{h_dcc.min():.4f}, {h_dcc.max():.4f}]')

In [ ]:
# Compute hedged returns and rolling hedge effectiveness
r_spot = ret_aligned['Spot'].values
r_fut  = ret_aligned['Futures'].values

# Hedged returns: r_hedged = r_spot - h * r_futures (using t-1 hedge ratio)
hedged_dcc = r_spot[1:] - h_dcc[:-1] * r_fut[1:]
hedged_ccc = r_spot[1:] - h_ccc[:-1] * r_fut[1:]
hedged_ols = r_spot[1:] - h_ols * r_fut[1:]
unhedged   = r_spot[1:]

dates = z_df.index[1:]

# Rolling HE (252-day window): HE = 1 - var(hedged) / var(unhedged)
window = 252

def rolling_he(hedged, unhedged, window):
    he = np.full(len(hedged), np.nan)
    for t in range(window, len(hedged)):
        var_h = np.var(hedged[t-window:t])
        var_u = np.var(unhedged[t-window:t])
        he[t] = 1 - var_h / var_u if var_u > 0 else np.nan
    return he

he_dcc = rolling_he(hedged_dcc, unhedged, window)
he_ccc = rolling_he(hedged_ccc, unhedged, window)
he_ols = rolling_he(hedged_ols, unhedged, window)

# Overall HE
print(f'Overall Hedge Effectiveness:')
print(f'  DCC:    {1 - np.var(hedged_dcc) / np.var(unhedged):.4f}')
print(f'  CCC:    {1 - np.var(hedged_ccc) / np.var(unhedged):.4f}')
print(f'  OLS:    {1 - np.var(hedged_ols) / np.var(unhedged):.4f}')

In [ ]:
# Chart 1: Hedge ratios over time
fig, ax = plt.subplots(figsize=(12, 4.5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.plot(z_df.index, h_dcc, color=MainBlue, linewidth=0.8, alpha=0.9,
        label='DCC hedge ratio')
ax.plot(z_df.index, h_ccc, color=IDAred, linewidth=0.8, alpha=0.9,
        label='CCC hedge ratio')
ax.axhline(y=h_ols, color='gray', linestyle='--', linewidth=1.0,
           label=f'OLS static ({h_ols:.3f})')

ax.set_ylabel('Hedge ratio')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=3, frameon=False)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_hedge_ratio.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_hedge_ratio.pdf')

In [ ]:
# Chart 2: Rolling hedge effectiveness
fig, ax = plt.subplots(figsize=(12, 4.5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.plot(dates, he_dcc, color=MainBlue, linewidth=1.0, label='DCC')
ax.plot(dates, he_ccc, color=IDAred, linewidth=1.0, label='CCC')
ax.plot(dates, he_ols, color='gray', linewidth=1.0, linestyle='--',
        label='OLS static')

# Shade crises
ax.axvspan(pd.Timestamp('2008-09-01'), pd.Timestamp('2009-06-30'),
           alpha=0.10, color='red', label='_nolegend_')
ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-06-30'),
           alpha=0.10, color='red', label='_nolegend_')
ax.text(pd.Timestamp('2008-10-01'), 0.98, 'GFC',
        fontsize=9, color='gray', style='italic')
ax.text(pd.Timestamp('2020-03-01'), 0.98, 'COVID',
        fontsize=9, color='gray', style='italic')

ax.set_ylabel('Hedge effectiveness (HE)')
ax.set_ylim(0.5, 1.02)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=3, frameon=False)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_hedge_effectiveness.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_hedge_effectiveness.pdf')

In [ ]:
# Hedge effectiveness comparison table
print('\nHedge Effectiveness Comparison')
print('=' * 55)

metrics = pd.DataFrame({
    'Method': ['DCC-GARCH', 'CCC-GARCH', 'OLS Static'],
    'Overall HE': [
        1 - np.var(hedged_dcc) / np.var(unhedged),
        1 - np.var(hedged_ccc) / np.var(unhedged),
        1 - np.var(hedged_ols) / np.var(unhedged)
    ],
    'Mean Hedge Ratio': [
        np.mean(h_dcc), np.mean(h_ccc), h_ols
    ],
    'Std Hedge Ratio': [
        np.std(h_dcc), np.std(h_ccc), 0.0
    ],
    'Hedged Vol (%)': [
        np.std(hedged_dcc) * np.sqrt(252),
        np.std(hedged_ccc) * np.sqrt(252),
        np.std(hedged_ols) * np.sqrt(252)
    ]
}).set_index('Method')

print(metrics.round(4).to_string())

## Results
- DCC-GARCH hedge ratio adapts to changing market conditions, varying significantly during crises
- During the GFC and COVID, hedge ratios deviate from the static OLS estimate, capturing basis risk dynamics
- DCC achieves the highest overall hedge effectiveness by accounting for time-varying correlations
- CCC performs similarly on average but misses short-term correlation shifts
- OLS static hedge is the simplest but least adaptive approach